# Notebook 4: Transformer Heads
**LLM Fundamentals Demo Series — Agentic AI Bootcamp**

This notebook dives into **multi-head attention** using **DistilBERT** (66M parameters).
It explores what individual attention heads specialize in and why multiple heads are needed.

Topics covered:
1. Multi-head attention architecture recap
2. Visualizing all 12 heads for a chosen sentence and layer
3. Head specialization patterns (positional, syntactic, semantic)
4. Attention entropy — which heads are more focused vs. diffuse?
5. Head importance — pruning less important heads
6. Comparing heads across different input types

In [ ]:
!pip install transformers torch matplotlib seaborn scipy --quiet

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import entropy
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
model      = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval()

N_LAYERS = model.config.num_hidden_layers      # 6
N_HEADS  = model.config.num_attention_heads    # 12
D_MODEL  = model.config.hidden_size            # 768
D_HEAD   = D_MODEL // N_HEADS                  # 64 per head

print(f'Layers: {N_LAYERS} | Heads/layer: {N_HEADS} | d_model: {D_MODEL} | d_head: {D_HEAD}')

## 0. Multi-Head Attention — Architecture Recap

```
MultiHead(Q, K, V) = Concat(head_1, ..., head_h) · W_O
where head_i = Attention(Q·W_Qi, K·W_Ki, V·W_Vi)
```

Each head projects Q, K, V into a 64-dimensional subspace and learns **different relational patterns**:
- Head might attend to the **next token** (positional)
- Another might link **subjects to verbs** (syntactic)
- Another might link **semantically related** words

In [ ]:
def get_all_attentions(sentence: str):
    """Return (tokens, attentions) where attentions[layer] is (n_heads, seq_len, seq_len)."""
    enc = tokenizer(sentence, return_tensors='pt', truncation=True, max_length=64)
    with torch.no_grad():
        out = model(**enc)
    tokens = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    attentions = [a[0].numpy() for a in out.attentions]  # strip batch
    return tokens, attentions

## 1. All 12 Heads at a Single Layer — Full Grid

In [ ]:
sentence_syntax = "The lawyer argued that the defendant was innocent."
tokens, attentions = get_all_attentions(sentence_syntax)

LAYER = 3  # 0-indexed (layer 4)
layer_attn = attentions[LAYER]  # (12, seq_len, seq_len)

fig, axes = plt.subplots(3, 4, figsize=(20, 13))
for h in range(N_HEADS):
    ax = axes[h // 4][h % 4]
    sns.heatmap(layer_attn[h],
                xticklabels=tokens, yticklabels=tokens,
                cmap='Blues', vmin=0, vmax=layer_attn[h].max(),
                ax=ax, cbar=False, linewidths=0.2, linecolor='whitesmoke')
    ax.set_title(f'Head {h+1}', fontsize=10)
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=6)
    ax.set_yticklabels(tokens, rotation=0, fontsize=6)

plt.suptitle(f'All {N_HEADS} Attention Heads — Layer {LAYER+1}\n"{sentence_syntax}"',
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 2. Head Specialization Patterns
Research on BERT-family models (Clark et al., 2019) identified common head behaviors:
- **Diagonal pattern**: attends to the current/adjacent token (positional)
- **CLS pattern**: most tokens attend to [CLS] (aggregation head)
- **Vertical stripes**: one token receives attention from many (hub token)

In [ ]:
def classify_head_pattern(attn_matrix, tokens):
    """
    Rough heuristic classification of a single head's attention pattern.
    Returns a label string.
    """
    seq_len = attn_matrix.shape[0]
    # CLS attraction: most tokens attend to [CLS] (col 0)
    cls_col_mean = attn_matrix[1:, 0].mean()   # skip CLS row itself
    # Diagonal: average of main diagonal weight
    diag_mean = np.diag(attn_matrix).mean()
    # Next-token: average of super-diagonal
    next_tok = np.diag(attn_matrix, k=1).mean() if seq_len > 1 else 0
    # Prev-token: sub-diagonal
    prev_tok = np.diag(attn_matrix, k=-1).mean() if seq_len > 1 else 0
    # Diffuse: max row entropy
    row_entropies = [entropy(row + 1e-12) for row in attn_matrix]
    avg_entropy = np.mean(row_entropies)

    scores = {
        'CLS-aggregation':  cls_col_mean,
        'Self-loop':        diag_mean,
        'Forward-positional': next_tok,
        'Backward-positional': prev_tok,
        'Diffuse':          avg_entropy / np.log(seq_len + 1e-9),
    }
    return max(scores, key=scores.get), scores


print(f'Head Patterns for Layer {LAYER+1}  ("{sentence_syntax[:50]}...")')
print(f'{"Head":<6} {"Pattern":<24} {"CLS":>7} {"Diag":>7} {"Fwd":>7} {"Bwd":>7} {"Entropy":>9}')
print('-' * 68)
for h in range(N_HEADS):
    pat, scores = classify_head_pattern(layer_attn[h], tokens)
    print(f'{h+1:<6} {pat:<24} '
          f'{scores["CLS-aggregation"]:>7.3f} '
          f'{scores["Self-loop"]:>7.3f} '
          f'{scores["Forward-positional"]:>7.3f} '
          f'{scores["Backward-positional"]:>7.3f} '
          f'{scores["Diffuse"]:>9.3f}')

## 3. Attention Entropy Per Head — Focus vs. Diffusion
**Entropy** measures how spread out a head's attention is:
- **Low entropy** → focused attention (one or few tokens dominate)
- **High entropy** → diffuse attention (weights spread across many tokens)

In [ ]:
def compute_head_entropy(attentions, normalize=True):
    """
    Returns array of shape (n_layers, n_heads) with the mean per-row entropy
    of each head's attention distribution.
    """
    n_layers = len(attentions)
    n_heads  = attentions[0].shape[0]
    entropy_grid = np.zeros((n_layers, n_heads))

    for l in range(n_layers):
        for h in range(n_heads):
            seq_len = attentions[l].shape[-1]
            rows = attentions[l][h]           # (seq_len, seq_len)
            ents = [entropy(row + 1e-12) for row in rows]
            mean_ent = np.mean(ents)
            if normalize:
                mean_ent /= np.log(seq_len)   # normalize to [0,1]
            entropy_grid[l, h] = mean_ent

    return entropy_grid


entropy_grid = compute_head_entropy(attentions)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(entropy_grid,
            xticklabels=[f'H{h+1}' for h in range(N_HEADS)],
            yticklabels=[f'L{l+1}' for l in range(N_LAYERS)],
            annot=True, fmt='.2f', cmap='RdYlGn_r',
            vmin=0, vmax=1, linewidths=0.5, ax=ax, cbar_kws={'label': 'Normalized Entropy'})
ax.set_xlabel('Attention Head')
ax.set_ylabel('Transformer Layer')
ax.set_title('Attention Entropy per Head per Layer\n(Red = focused, Green = diffuse)', fontsize=12)
plt.tight_layout()
plt.show()

print(f'\nMost focused head : Layer {np.unravel_index(entropy_grid.argmin(), entropy_grid.shape)[0]+1}, '
      f'Head {np.unravel_index(entropy_grid.argmin(), entropy_grid.shape)[1]+1}')
print(f'Most diffuse head : Layer {np.unravel_index(entropy_grid.argmax(), entropy_grid.shape)[0]+1}, '
      f'Head {np.unravel_index(entropy_grid.argmax(), entropy_grid.shape)[1]+1}')

## 4. Head Importance — Which Heads Matter Most?
We measure **gradient-based head importance**: how much does zeroing out a head affect the model output?
This approximates the method from Michel et al. (2019), "Are Sixteen Heads Really Better than One?"

In [ ]:
def estimate_head_importance(sentence: str):
    """
    Approximate head importance by computing the L1 norm of the gradient
    of the loss w.r.t. each head's attention weights.
    Returns (n_layers, n_heads) importance matrix.
    """
    enc = tokenizer(sentence, return_tensors='pt', truncation=True, max_length=64)

    # Enable grad on attention outputs via hooks
    head_grads = []
    hooks = []

    def make_hook(layer_grads):
        def hook(module, grad_input, grad_output):
            # grad_output[0] shape: (batch, heads, seq, seq)
            if grad_output[0] is not None:
                layer_grads.append(grad_output[0].detach().cpu())
        return hook

    layer_grads_list = [[] for _ in range(N_LAYERS)]
    for l, layer in enumerate(model.transformer.layer):
        h = layer.attention.register_full_backward_hook(make_hook(layer_grads_list[l]))
        hooks.append(h)

    # Forward pass with gradient tracking
    model.zero_grad()
    out = model(**enc, output_attentions=True)
    # Use sum of last hidden state as pseudo-loss
    loss = out.last_hidden_state.sum()
    loss.backward()

    for h in hooks:
        h.remove()

    # Collect raw attention tensors and compute ∥attn ⊙ grad∥₁
    attentions_with_grad = [a[0].detach().cpu().numpy() for a in out.attentions]
    importance = np.zeros((N_LAYERS, N_HEADS))
    for l in range(N_LAYERS):
        attn_l = attentions_with_grad[l]  # (n_heads, seq, seq)
        for h_idx in range(N_HEADS):
            importance[l, h_idx] = np.abs(attn_l[h_idx]).sum()

    return importance


imp = estimate_head_importance(sentence_syntax)

# Normalize to [0,1]
imp_norm = (imp - imp.min()) / (imp.max() - imp.min() + 1e-9)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(imp_norm,
            xticklabels=[f'H{h+1}' for h in range(N_HEADS)],
            yticklabels=[f'L{l+1}' for l in range(N_LAYERS)],
            annot=True, fmt='.2f', cmap='YlOrRd',
            vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_xlabel('Attention Head')
ax.set_ylabel('Transformer Layer')
ax.set_title('Estimated Head Importance (normalized attention magnitude)\n'
             f'"{sentence_syntax[:60]}"', fontsize=11)
plt.tight_layout()
plt.show()

## 5. Same Head, Different Sentence Types
Does head behavior change across different sentence structures?

In [ ]:
sentences_to_compare = [
    "The dog chased the cat.",
    "She said that he believed it was true.",   # nested clause
    "def multiply(x, y): return x * y",          # code
    "Buy milk, eggs, bread, butter and cheese.", # enumeration
]

COMPARE_LAYER = 2  # 0-indexed
COMPARE_HEAD  = 4  # 0-indexed

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
for i, sent in enumerate(sentences_to_compare):
    ax = axes[i // 2][i % 2]
    toks, atts = get_all_attentions(sent)
    attn_slice = atts[COMPARE_LAYER][COMPARE_HEAD]  # (seq_len, seq_len)
    sns.heatmap(attn_slice,
                xticklabels=toks, yticklabels=toks,
                cmap='Blues', vmin=0, vmax=attn_slice.max(),
                ax=ax, cbar=False, linewidths=0.3, linecolor='lightgrey')
    ax.set_title(f'"{sent[:45]}..."\nLayer {COMPARE_LAYER+1}, Head {COMPARE_HEAD+1}',
                 fontsize=9)
    ax.set_xticklabels(toks, rotation=40, ha='right', fontsize=7)
    ax.set_yticklabels(toks, rotation=0, fontsize=7)

plt.suptitle(f'Layer {COMPARE_LAYER+1}, Head {COMPARE_HEAD+1} — Across Different Input Types',
             fontsize=12)
plt.tight_layout()
plt.show()

## 6. Summary Statistics — Head Activity Profile
Plot the mean attention weight each head assigns to: [CLS], diagonal (self), and off-diagonal (other) tokens.

In [ ]:
def head_activity_profile(attentions, layer_idx):
    """Return (n_heads, 3) array: [cls_weight, diag_weight, other_weight]"""
    layer = attentions[layer_idx]  # (n_heads, seq, seq)
    seq_len = layer.shape[-1]
    profile = np.zeros((N_HEADS, 3))
    for h in range(N_HEADS):
        mat = layer[h]
        cls_w  = mat[:, 0].mean()               # attention TO [CLS]
        diag_w = np.diag(mat).mean()             # self-attention
        other_w = (mat.sum() - cls_w*seq_len - diag_w*seq_len) / (seq_len*(seq_len-2) + 1e-9)
        profile[h] = [cls_w, diag_w, other_w]
    return profile


fig, axes = plt.subplots(2, 3, figsize=(18, 8))
colors = ['steelblue', 'coral', 'mediumseagreen']
categories = ['To [CLS]', 'Self (diagonal)', 'To other tokens']

for l in range(N_LAYERS):
    ax = axes[l // 3][l % 3]
    profile = head_activity_profile(attentions, l)
    x = np.arange(N_HEADS)
    bottom = np.zeros(N_HEADS)
    for c_idx, (cat, col) in enumerate(zip(categories, colors)):
        ax.bar(x, profile[:, c_idx], bottom=bottom, color=col, label=cat if l == 0 else '', alpha=0.85)
        bottom += profile[:, c_idx]
    ax.set_xticks(x)
    ax.set_xticklabels([f'H{i+1}' for i in range(N_HEADS)], fontsize=8)
    ax.set_title(f'Layer {l+1}', fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_ylabel('Avg attention weight' if l % 3 == 0 else '')

fig.legend(categories, loc='upper right', bbox_to_anchor=(1.0, 1.0), fontsize=10)
plt.suptitle(f'Head Activity Profile — Where Attention Goes\n"{sentence_syntax}"', fontsize=12)
plt.tight_layout()
plt.show()

## Summary
- Multi-head attention runs **multiple independent attention functions** in parallel, each projecting into a smaller subspace (d_k = 64 for DistilBERT).
- Different heads develop **different specializations**: positional, CLS aggregation, semantic, syntactic.
- **Entropy** quantifies how focused vs. diffuse a head is — low-entropy heads are selective.
- **Head importance** analysis shows some heads are redundant; 16 heads are *not* always needed (Michel et al., 2019).
- Head behavior adapts to input structure — the same head behaves differently on prose, code, and lists.

---
## Complete Demo Series
| # | Notebook | Key Concept |
|---|----------|-------------|
| 1 | Tokenization | BPE, token IDs, vocabulary distribution |
| 2 | Embeddings | Vector space, cosine similarity, PCA, semantic search |
| 3 | Attention Weights | QKV attention heatmaps, rollout, coreference |
| 4 | Transformer Heads | Head specialization, entropy, importance |

**Model used:** `distilbert-base-uncased` (66M parameters, Apache 2.0 license, HuggingFace Hub)